In [18]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import plotly.express as px
# Import the processing module from the same folder
from processing import load_solutions, add_kwargs_as_indices, combine_solutions, read_parquet_and_convert, add_fields
# import processing 
# from pivottablejs import pivot_ui
G_save = True

In [19]:
def create_envelope(s_ed, s_uc, group_by = ['configuration', 'µ', 'iteration', 'day', 'hour,', 'r_id']):
    # Copy the relevant columns from s_ed['storage']
    envelope = s_ed['storage'][group_by + ['SOE_MWh', 'envelope_up_MWh', 'envelope_down_MWh']].copy()

    # Perform the first left join
    envelope = envelope.merge(
        s_uc['storage'][[col for col in group_by if col != 'iteration'] + ['SOE_MWh', 'envelope_up_MWh', 'envelope_down_MWh']].rename(
            columns={'SOE_MWh': 'SOE_DA_MWh', 'envelope_up_MWh': 'envelope_up_DA_MWh', 'envelope_down_MWh': 'envelope_down_DA_MWh'}
        ),
        on=[col for col in group_by if col != 'iteration'],
        how='left'
    )
    
    # Perform the second left join
    envelope = envelope.merge(
        s_uc['storage_parameters'][['r_id', 'SOE_max_MWh', 'initial_energy_proportion']].drop_duplicates(),
        on='r_id',
        how='left'
    )
    
    # Calculate initial state of energy (SOE) based on maximum SOE and initial energy proportion
    envelope['SOE_0_MWh'] = envelope['SOE_max_MWh'] * envelope['initial_energy_proportion']
    envelope['SOC'] = envelope['SOE_MWh'] / envelope['SOE_max_MWh'] 
    # Group by day, configuration, and resource ID, and get the last entry for each group

    return envelope

In [20]:

ss = [
    # {'solution_folder': f"RTS-GMLC_v5.2.2s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2z", 'model_type' : 'envelope'}, # last opertional
    # {'solution_folder': f"RTS-GMLC_v5.2.2.1s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2.3z", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2.3.1z", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2.4z", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2.4.1z", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v6.2.2s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v15.0s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v16.0.1su", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v16.0.2su", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v16.1su", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v16.2su", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v18.0.3s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v18.3s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_envelope_benchmark_v5.0s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_envelope_compare_v6.0s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v9.0su", 'VLGEN': 30, 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v19.3s", 'VLGEN': 30, 'model_type' : 'e-reserve'},
    {'solution_folder': f"RTS-GMLC_v19.0.3s", 'VLGEN': 30, 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v6.2.3s", 'model_type' : 'e-reserve'}
    ]

s_uc = []
s_ed = []
gcd_KPI_adequacy = []
gcdi_KPI_adequacy = []
for sol in ss:
    # ρ = sol['ρ']
    s = sol['solution_folder']
    s_uc_name = 's_suc' if sol['model_type'] == 'stochastic' else 's_uc'
    # s_uc_ = load_solutions(s_uc_name, os.path.join("..", "output", s), [7], model_type = sol['model_type'], ρ=ρ, solution_id = s)
    # s_ed_ = load_solutions("s_ed", os.path.join("..", "output", s), [7], model_type = sol['model_type'], ρ=ρ, solution_id = s)
    # s_uc.append(s_uc_)
    # s_ed.append(s_ed_)

    gcd_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcd_KPI_adequacy.parquet"))
    gcd_KPI_adequacy_ = add_fields(gcd_KPI_adequacy_, model_type = sol['model_type'], solution_id = s) 

    gcdi_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcdi_KPI_adequacy.parquet"))
    gcdi_KPI_adequacy_ = add_fields(gcdi_KPI_adequacy_, model_type = sol['model_type'], solution_id = s)

    gcd_KPI_adequacy.append(gcd_KPI_adequacy_)
    gcdi_KPI_adequacy.append(gcdi_KPI_adequacy_)

# s_uc = combine_solutions(s_uc)
# s_ed = combine_solutions(s_ed)
gcd_KPI_adequacy = pd.concat(gcd_KPI_adequacy)
gcdi_KPI_adequacy = pd.concat(gcdi_KPI_adequacy)

if 'µ' in gcdi_KPI_adequacy.columns: 
#         # out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)
    gcdi_KPI_adequacy['model_type'] = gcdi_KPI_adequacy.apply(lambda x: 'conservative' if ('envelope' in x['model_type']) & (x['µ'] == 1) else x['model_type'], axis=1)
    gcd_KPI_adequacy['model_type'] = gcd_KPI_adequacy.apply(lambda x: 'conservative' if ('envelope' in x['model_type']) & (x['µ'] == 1) else x['model_type'], axis=1)




../output/RTS-GMLC_v18.0.3s/all_gcd_KPI_adequacy.parquet
../output/RTS-GMLC_v18.0.3s/all_gcdi_KPI_adequacy.parquet
../output/RTS-GMLC_v18.3s/all_gcd_KPI_adequacy.parquet


../output/RTS-GMLC_v18.3s/all_gcdi_KPI_adequacy.parquet
../output/RTS-GMLC_v19.3s/all_gcd_KPI_adequacy.parquet
../output/RTS-GMLC_v19.3s/all_gcdi_KPI_adequacy.parquet
../output/RTS-GMLC_v19.0.3s/all_gcd_KPI_adequacy.parquet
../output/RTS-GMLC_v19.0.3s/all_gcdi_KPI_adequacy.parquet


In [21]:
gcdi_KPI_adequacy['model_type'].unique()

array(['envelope', 'conservative', 'e-reserve'], dtype=object)

In [22]:
if G_save:
    out= gcd_KPI_adequacy.copy()
    if 'µ' in out.columns: 
        out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)
        # out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: ((x[0] =='envelope')*(x[1]==1)*'conservatice' + x[0]), axis = 1)

    renames = {'µ': 'mu', 'ρ' : 'rho'}
    renames = {k: v for k, v in renames.items() if k in out.columns}
    out.rename(columns = renames, inplace = True)
    out.to_csv('gcd_KPI_adequacy.csv', index=False)
    gcdi_KPI_adequacy.rename(columns = renames, inplace = True)
    gcdi_KPI_adequacy.reset_index().to_csv('gcdi_KPI_adequacy.csv', index=False)

/tmp/ipykernel_1379/3797213802.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)


In [23]:
gcdi_KPI_adequacy

,model_type,solution_id,configuration,day,iteration,LLD_h,ENS_MWh,input_load_MWh,CURD_h,CUR_MWh,...,slack_energy_reserve_down_uc_MWh,required_energy_reserve_up_uc_MWh,slack_energy_reserve_up_uc_MWh,required_energy_reserve_down_uc_MWh,energy_reserve_down_uc_MWh,energy_reserve_up_uc_MWh,thermal_energy_reserve_down_uc_MWh,thermal_energy_reserve_up_uc_MWh,storage_energy_reserve_down_uc_MWh,storage_energy_reserve_up_uc_MWh
0,envelope,RTS-GMLC_v18.0.3s,base_ramp_storage_envelopes_up_0_54_dn_0_54,1,demand_1,0,3.592520e-13,53034.826584,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,conservative,RTS-GMLC_v18.0.3s,base_ramp_storage_envelopes_up_1_dn_1,1,demand_1,0,1.889027e-11,53034.826584,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,conservative,RTS-GMLC_v18.0.3s,base_ramp_storage_envelopes_up_1_dn_1,2,demand_1,0,9.121942e-13,43036.274891,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,envelope,RTS-GMLC_v18.0.3s,base_ramp_storage_envelopes_up_0_49_dn_0_49,2,demand_1,0,1.456518e-11,43036.274891,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,envelope,RTS-GMLC_v18.0.3s,base_ramp_storage_envelopes_up_0_63_dn_0_63,3,demand_1,0,2.237584e-12,41727.386042,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,e-reserve,RTS-GMLC_v19.0.3s,base_ramp_storage_envelopes_up_1_dn_1,96,demand_1,0,1.507639e-11,45763.422215,0,0.0,...,1.103919e-09,1.082388e+06,1.103919e-09,1.610019e+06,1.610020e+06,1.082389e+06,306810.379282,9.390414e+04,1.303209e+06,9.884853e+05
96,e-reserve,RTS-GMLC_v19.0.3s,base_ramp_storage_envelopes_up_1_dn_1,97,demand_1,0,1.018498e-16,60791.847339,0,0.0,...,9.099882e-10,1.370684e+06,9.099882e-10,1.789586e+06,1.789586e+06,1.370685e+06,495638.199505,1.556891e+05,1.293948e+06,1.214996e+06
97,e-reserve,RTS-GMLC_v19.0.3s,base_ramp_storage_envelopes_up_1_dn_1,98,demand_1,0,1.090026e-17,68771.700672,0,0.0,...,2.473038e-10,8.769593e+05,2.472773e-10,1.398994e+06,1.398994e+06,8.769596e+05,508720.408862,1.728280e+05,8.902740e+05,7.041315e+05
98,e-reserve,RTS-GMLC_v19.0.3s,base_ramp_storage_envelopes_up_1_dn_1,99,demand_1,0,4.896855e-11,54755.193058,0,0.0,...,7.435515e-10,1.182059e+06,7.435515e-10,1.544832e+06,1.544832e+06,1.182060e+06,412049.675329,1.405649e+05,1.132782e+06,1.041495e+06
